In [1]:
import tiktoken
from torch.utils.data import Dataset,DataLoader
import torch

In [3]:
### initing random ids
input_token_ids=torch.tensor([2,3,5])

In [4]:
vocabulary_size=10
n_dim=6 ## dimention of embedding

In [5]:
embedding=torch.nn.Embedding(vocabulary_size,n_dim)
embedding.weight.size()

torch.Size([10, 6])

In [6]:
## lookup table for 10 words
embedding.weight

Parameter containing:
tensor([[ 1.0618e+00, -9.3630e-01, -1.6945e+00,  1.7869e+00, -1.5433e+00,
          8.8833e-01],
        [-2.6579e-02,  6.2499e-01, -1.2802e+00, -9.6982e-01,  1.1504e-03,
          2.3995e-01],
        [ 9.8959e-01,  4.2131e-01,  1.0507e+00, -4.0970e-01,  1.0757e+00,
         -1.6874e-01],
        [-1.5400e+00, -1.0113e+00, -1.6319e+00,  6.1480e-01, -1.0155e-01,
         -1.5291e+00],
        [-5.4240e-01,  2.7995e-01,  5.6540e-01,  2.9238e-01,  1.2239e+00,
         -9.1640e-01],
        [ 7.4684e-02, -1.2749e+00, -1.1769e+00,  1.4350e+00, -1.2428e+00,
         -1.1524e+00],
        [ 6.9044e-01,  9.6927e-01,  1.7436e+00,  9.2483e-01, -1.6540e-01,
         -3.5673e-01],
        [ 1.4854e+00, -1.9307e+00,  2.3686e-02,  3.1639e-01,  1.6606e-01,
          1.1654e+00],
        [ 9.7655e-01,  4.6756e-01,  2.7505e-01, -1.2018e+00, -2.2246e+00,
         -3.1088e-02],
        [ 9.0359e-01,  9.7435e-01, -9.6507e-01,  9.2562e-01,  3.3216e-01,
         -3.4351e-01]], require

In [7]:
embedding.weight[input_token_ids]

tensor([[ 0.9896,  0.4213,  1.0507, -0.4097,  1.0757, -0.1687],
        [-1.5400, -1.0113, -1.6319,  0.6148, -0.1016, -1.5291],
        [ 0.0747, -1.2749, -1.1769,  1.4350, -1.2428, -1.1524]],
       grad_fn=<IndexBackward0>)

In [ ]:
## lets make it practical
vocabulary_size=50257 ## max limit for tokenizer for gpt2
n_dim=512
vector_emd=torch.nn.Embedding(vocabulary_size,n_dim)

In [10]:
class CustomDataset(Dataset):
    def __init__(self,raw_text,context_size,tokenizer,strides):
        self.input_ids=[]
        self.output_ids=[]
        tokens=tokenizer.encode(raw_text,allowed_special={"<|endoftext|>"})
        for i in range(0,len(tokens)-context_size,strides):
            input_chunk=tokens[i:i+context_size]
            output_chunk=tokens[i+1:i+1+context_size]
            self.input_ids.append(torch.tensor(input_chunk))
            self.output_ids.append(torch.tensor(output_chunk))

    def __len__(self):
        return len(self.input_ids)    
    def __getitem__(self, index):
        return self.input_ids[index],self.output_ids[index]    
        

In [11]:
with open("the-verdict.txt","r") as file:
    text=file.read()

In [ ]:
def create_dataloader(txt, batch_size=4, max_length=256, 
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):
   ## here max_length=context_size

    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = CustomDataset(txt,context_size=max_length,tokenizer=tokenizer, strides=stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader